## Creating Messaging Resources

## Core Messaging Concepts in Google Cloud Pub/Sub

Hey there, welcome back! In our introductory discussion, we briefly touched on the core messaging services available—fully managed solutions perfect for decoupling and scaling microservices, distributed systems, and serverless applications. While we haven't delved into specifics yet, it's important to note that these services offer different types of messaging resources, each with its own features and behaviors. Let's explore these in more detail now:

* **Topics:** Named resource channels where publishers explicitly send message data.
* **Subscriptions:** Named resource endpoints attached to a specific topic that pull or receive message streams.

This decoupling allows different components of your application to communicate asynchronously without requiring direct point-to-point connections.

---

## Key Configuration Options for Messaging Resources

Let's delve into some more technical aspects. Messaging resources have a set of characteristics, known as configuration options, that control their runtime behavior. Here are some critical parameters you should be aware of:

* **Acknowledgment Deadline (Subscriptions):** This setting determines how long a message remains "invisible" (locked) to other subscriber workers after it is initially delivered. If your consumer fails to acknowledge (`ack()`) the payload within this window, Pub/Sub makes it available for redelivery. The default is 10 seconds, but it can be manually extended up to 600 seconds (10 minutes).
* **Message Retention Duration (Topics/Subscriptions):** Specifies how long unacknowledged data persists in the message queue pool. The default is 7 days, but it can be customized within a window ranging from 10 minutes up to 7 days.
* **Dead Letter Topic (Subscriptions):** A specialized routing fallback topic where messages that repeatedly fail processing loops after a designated number of delivery attempts are quarantined for independent debugging.
* **Filter Expression (Subscriptions):** An attribute-matching mechanism that evaluates message metadata server-side, ensuring subscriptions only pull items that match explicit filtering conditions.

By adjusting these options when creating your messaging resources, you can optimize their behavior for various use cases. For instance, setting a short acknowledgment deadline reduces the time for a stalled message to become available to other subscribers, whereas a longer message retention duration ensures important messages persist longer if unprocessed.

---

## Creating a Pub/Sub Topic and Subscription

Let's begin with some practical coding! Our first task is creating a simple topic and subscription. The Google Cloud Python client libraries make this process straightforward.

```python
from google.cloud import pubsub_v1

project_id = "your-project-id"
topic_id = "simple-topic"
subscription_id = "simple-subscription"

# 1. Initialize Clients
publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()

# 2. Setup and Provision Topic
topic_path = publisher.topic_path(project_id, topic_id)
topic = publisher.create_topic(request={"name": topic_path})
print("Topic created:", topic.name)

# 3. Setup and Provision Subscription Attached to Topic
subscription_path = subscriber.subscription_path(project_id, subscription_id)
subscription = subscriber.create_subscription(
    request={
        "name": subscription_path,
        "topic": topic_path,
    }
)
print("Subscription created:", subscription.name)

```

**Console Output:**

> `Topic created: projects/your-project-id/topics/simple-topic`
> `Subscription created: projects/your-project-id/subscriptions/simple-subscription`

In this script, we're importing the necessary modules, initializing the publisher and subscriber clients, and creating a topic and a subscription. The `create_topic` and `create_subscription` methods return the created resource blueprints, and we print out their unique global paths.

---

## Configuring Subscription Attributes

You can customize your subscription by setting specific attributes during its initial creation. Here is an example of a subscription configured with an acknowledgment deadline of 45 seconds and a message retention duration of 1 day (86,400 seconds):

```python
from google.cloud import pubsub_v1

project_id = "your-project-id"
topic_id = "attribute-config-topic"
subscription_id = "attribute-config-subscription"

# Initialize Client Endpoints
publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()

# Set up the base Topic
topic_path = publisher.topic_path(project_id, topic_id)
topic = publisher.create_topic(request={"name": topic_path})
print("Topic created:", topic.name)

# Set up the Subscription with custom delivery traits
subscription_path = subscriber.subscription_path(project_id, subscription_id)
subscription = subscriber.create_subscription(
    request={
        "name": subscription_path,
        "topic": topic_path,
        "ack_deadline_seconds": 45,
        "message_retention_duration": {"seconds": 86400},
    }
)

print("Subscription with configured attributes created:", subscription.name)
print("Acknowledgment deadline:", subscription.ack_deadline_seconds, "seconds")
print("Message retention duration:", subscription.message_retention_duration.seconds, "seconds")

```

**Console Output:**

> `Topic created: projects/your-project-id/topics/attribute-config-topic`
> `Subscription with configured attributes created: projects/your-project-id/subscriptions/attribute-config-subscription`
> `Acknowledgment deadline: 45 seconds`
> `Message retention duration: 86400 seconds`

These properties instantly alter the lifecycle behavior of messages inside the subscription, allowing you to tailor resource resource-handling thresholds to match your backend computing capacities.

---

## Lesson Summary and Upcoming Practices

Great job! You've now learned how to create and configure messaging resources with different options for message delivery and retention. Be sure to practice these skills in the upcoming exercises, where you'll be asked to create and configure topics and subscriptions on your own.

In the next unit, we'll move on to **"Sending and Receiving Messages."** Remember, practice makes perfect. Keep coding, and I'll see you in the next lesson!

## Creating Google Cloud Pub/Sub Topics and Subscriptions with Different Configurations

For this task, all you need to do is run the provided Python script that creates different types of Google Cloud Pub/Sub topics and subscriptions: a simple topic/subscription pair, a topic/subscription with various attributes, and an ordered topic/subscription pair. No extra coding is required on your part. Your objective is to understand the process of creating messaging resources with different configurations and to observe the differences in their respective names and configurations.

Important Note: Running scripts can modify the resources in our GCP simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
from google.cloud import pubsub_v1
import os

# Initialize clients
publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()
project_id = os.getenv('GOOGLE_CLOUD_PROJECT', 'your-project-id')

# Simple topic and subscription
topic_path = publisher.topic_path(project_id, 'SimpleTopic')
subscription_path = subscriber.subscription_path(project_id, 'SimpleSubscription')

topic = publisher.create_topic(request={"name": topic_path})
print("Simple Topic name:", topic.name)

subscription = subscriber.create_subscription(
    request={"name": subscription_path, "topic": topic_path}
)
print("Simple Subscription name:", subscription.name)

# Topic and subscription with several attributes
topic_path = publisher.topic_path(project_id, 'AttributeConfigTopic')
subscription_path = subscriber.subscription_path(project_id, 'AttributeConfigSubscription')

topic = publisher.create_topic(request={"name": topic_path})
print("Topic with attributes name:", topic.name)

subscription = subscriber.create_subscription(
    request={
        "name": subscription_path,
        "topic": topic_path,
        "ack_deadline_seconds": 45,  # equivalent to VisibilityTimeout
        "message_retention_duration": {"seconds": 86400},  # 1 day retention
        "enable_message_ordering": False
    }
)
print("Subscription with attributes name:", subscription.name)

# Ordered topic and subscription (equivalent to FIFO)
topic_path = publisher.topic_path(project_id, 'OrderedTopic')
subscription_path = subscriber.subscription_path(project_id, 'OrderedSubscription')

topic = publisher.create_topic(request={"name": topic_path})
print("Ordered Topic name:", topic.name)

subscription = subscriber.create_subscription(
    request={
        "name": subscription_path,
        "topic": topic_path,
        "enable_message_ordering": True  # equivalent to FIFO ordering
    }
)
print("Ordered Subscription name:", subscription.name)

```

Running this script highlights the immense flexibility of Google Cloud Pub/Sub’s topology. By adjusting a few keys in your creation payload, you completely transform how messages are buffered, ordered, and re-routed across your system.

Here is an architectural breakdown of the three distinct messaging patterns executed by the script:

---

## 1. Simple Configuration (Default Standard Queue)

```python
subscription = subscriber.create_subscription(
    request={"name": subscription_path, "topic": topic_path}
)

```

* **The Blueprint:** This provisions a standard baseline subscription.
* **Under the Hood:** By omitting configuration flags, it inherits Pub/Sub default profiles: a **10-second** acknowledgment deadline and a maximum **7-day** message retention buffer. It handles massive throughput over parallel processing threads but makes no hard guarantees about delivery order.

---

## 2. Resource Management Configuration (SLA Adjustments)

```python
"ack_deadline_seconds": 45,
"message_retention_duration": {"seconds": 86400}  # 1 day

```

* **The Blueprint:** This builds a message track engineered around system performance and resource costs.
* **Under the Hood:** Extending `ack_deadline_seconds` to **45 seconds** gives your backend workers more time to process resource-heavy tasks (like generating reports or resizing files) before Pub/Sub assumes the worker crashed and attempts a redelivery. Dropping `message_retention_duration` down to **1 day** (86,400 seconds) prevents obsolete data from piling up in the queue, saving space and optimizing maintenance loops.

---

## 3. Strict First-In, First-Out Ordering (FIFO)

```python
"enable_message_ordering": True

```

* **The Blueprint:** This changes the delivery mechanics to guarantee sequential data consumption.
* **Under the Hood:** By default, Pub/Sub values absolute throughput over delivery order. Enabling `enable_message_ordering` shifts it into a strict **FIFO (First-In, First-Out)** engine for messages that share the same `ordering_key`.
* **The Catch:** If an early message fails processing, Pub/Sub pauses delivery for that specific ordering key until the stuck message is successfully acknowledged or its deadline expires, ensuring state changes are applied in the exact sequence they happened.

---

### Configuration Blueprint Reference

| Configuration Type | Acknowledgment Window | Retention Limit | Delivery Mode | Ideal For |
| --- | --- | --- | --- | --- |
| **Simple Standard** | 10 Seconds | 7 Days | Best-Effort Order | High-volume log aggregations, generic user notifications. |
| **Attribute Modified** | 45 Seconds | 1 Day | Best-Effort Order | Intensive compute tasks with high processing latency. |
| **Strict Ordered** | 10 Seconds | 7 Days | **Strict FIFO** | Financial ledgers, seat bookings, inventory updates. |

## Configure Google Cloud Pub/Sub Subscription Attributes

In this task, you will modify an existing Python script. The script creates a Google Cloud Pub/Sub topic and subscription with pre-set attributes. Your goal is to understand the original subscription configuration and adjust some attributes according to new technical requirements.

Consider the following:

    The processing time for messages usually takes up to 10 minutes, so messages should have an acknowledgment deadline for that duration (ack_deadline_seconds).
    Messages should remain in the subscription for 3 days (message_retention_duration).
    Enable message ordering for better processing control (enable_message_ordering).

Important Note: Running scripts can modify the resources in our GCP simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, remember that resetting will erase any code changes. If you've made any modifications that you'd like to keep, be sure to copy your code to your clipboard before hitting reset.

```python
from google.cloud import pubsub_v1
import os

# Initialize clients
publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()
project_id = os.getenv('GOOGLE_CLOUD_PROJECT', 'your-project-id')

# Topic and subscription with configured attributes
topic_path = publisher.topic_path(project_id, 'AttributeConfigTopic')
subscription_path = subscriber.subscription_path(project_id, 'AttributeConfigSubscription')

topic = publisher.create_topic(request={"name": topic_path})
print("Topic created:", topic.name)

subscription = subscriber.create_subscription(
    request={
        "name": subscription_path,
        "topic": topic_path,
        "ack_deadline_seconds": 45,
        "message_retention_duration": {"seconds": 86400},
        "enable_message_ordering": False
    }
)
print(f"Subscription configured with initial settings: {subscription.name}")
```

To meet the new architectural requirements, we need to convert your business logic parameters into structural configurations that the Pub/Sub API expects:

1. **Acknowledgment Deadline:** 10 minutes equals **$600$ seconds**.
2. **Retention Window:** 3 days translates to **$259,200$ seconds** ($3 \text{ days} \times 24 \text{ hours} \times 60 \text{ minutes} \times 60 \text{ seconds}$).
3. **Sequential Processing:** Toggle `enable_message_ordering` from `False` to `True`.

Here is the fully adjusted and updated Python script:

```python
from google.cloud import pubsub_v1
import os

# Initialize clients
publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()
project_id = os.getenv('GOOGLE_CLOUD_PROJECT', 'your-project-id')

# Topic and subscription with configured attributes
topic_path = publisher.topic_path(project_id, 'AttributeConfigTopic')
subscription_path = subscriber.subscription_path(project_id, 'AttributeConfigSubscription')

topic = publisher.create_topic(request={"name": topic_path})
print("Topic created:", topic.name)

# --- TODO FIXED: Adjusted configuration parameters to meet new technical requirements ---
subscription = subscriber.create_subscription(
    request={
        "name": subscription_path,
        "topic": topic_path,
        "ack_deadline_seconds": 600,                    # 10 minutes processing window
        "message_retention_duration": {"seconds": 259200},  # 3 days buffer duration
        "enable_message_ordering": True                 # Strict message ordering activated
    }
)

print(f"Subscription configured with updated settings: {subscription.name}")
print(f"Verified Ack Deadline: {subscription.ack_deadline_seconds} seconds")
print(f"Verified Retention Window: {subscription.message_retention_duration.seconds} seconds")
print(f"Verified Message Ordering Status: {subscription.enable_message_ordering}")

```

---

### Deep Dive: Behind-the-Scenes Mechanics

* **Maximum Invisible Cap (`ack_deadline_seconds: 600`):** Setting this to 600 reaches the absolute ceiling for standard Pub/Sub subscription acknowledgement boundaries. If your backend consumer worker experiences a deep computing stall that lasts 601 seconds without extending the lease, the server assumes consumer death and makes the payload available for processing by a separate sibling worker.
* **The Ordering Catch (`enable_message_ordering: True`):** Simply toggling this parameter on is step one. To enforce strict sequence parsing at runtime, your publishing engine must explicitly attach an `ordering_key` metadata string to each outgoing `pubsub_v1.PubsubMessage` payload. Messages that share the exact same key will be consumed sequentially in the precise order they hit the ingest endpoint.

## Debug Google Cloud Pub/Sub Topic Creation Script

Welcome to this task! The existing Python script is designed to create a topic in Google Cloud Pub/Sub but seemingly contains some errors preventing the topic from being created correctly. Your goal in this task is to sift through this script, identify the bug or bugs causing the problem, and fix them. After correcting the errors, run your updated script to verify the successful creation of the Pub/Sub topic.

Important Note: Running scripts can modify the resources in our GCP simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, remember that resetting will erase any code changes. To preserve your code during a reset, consider copying it to your clipboard.

```python
from google.cloud import pubsub_v1
import os

# Initialize publisher client
publisher = pubsub_v1.PublisherClient()
project_id = os.getenv('GOOGLE_CLOUD_PROJECT', 'your-project-id')

# Topic with reserved prefix (invalid naming)
topic_id = 'googAttributeConfigTopic'
topic_path = publisher.topic_path(project_id, topic_id)

topic = publisher.create_topic(request={"name": topic_path})
print("Topic created:", topic.name)

```

The issue causing this script to crash is a violation of Google Cloud Pub/Sub's naming rules.

Specifically, you cannot start a topic or subscription name with the string **`goog`**. This prefix is strictly reserved by Google Cloud for internal system infrastructure routing hooks. If a client script attempts to register a resource starting with those letters, the backend validation layer instantly rejects it with an `InvalidArgument` gRPC exception.

---

### Fixed Python Script

To fix this, we simply need to remove or change the forbidden prefix. Let's update `googAttributeConfigTopic` to a valid identifier like `AttributeConfigTopic`:

```python
from google.cloud import pubsub_v1
import os

# Initialize publisher client
publisher = pubsub_v1.PublisherClient()
project_id = os.getenv('GOOGLE_CLOUD_PROJECT', 'your-project-id')

# --- TODO FIXED: Removed the restricted 'goog' prefix from the topic ID ---
topic_id = 'AttributeConfigTopic'
topic_path = publisher.topic_path(project_id, topic_id)

# Create the topic with a valid, compliant naming scheme
topic = publisher.create_topic(request={"name": topic_path})
print("Topic created successfully:", topic.name)

```

---

### Google Cloud Pub/Sub Resource Naming Rules

When naming your Pub/Sub resources, keep these four strict constraints in mind to prevent unexpected API rejections:

* **Forbidden Prefixes:** Must **not** begin with the letters `goog`.
* **Allowed Characters:** Restricted entirely to alphanumeric characters, hyphens (`-`), underscores (`_`), periods (`.`), tildes (`~`), plus signs (`+`), and percent signs (`%`).
* **Length Boundaries:** Must contain between **$3$ and $255$ characters**.
* **Global Uniqueness:** Topic names must be completely unique within your specific Google Cloud project container boundaries.

## Google Cloud Pub/Sub Subscription with Acknowledgment Deadline Configuration

## Google Cloud Pub/Sub Configuration for Effective Message Processing

## Google Cloud Pub/Sub Configuration for Effective Message Processing